### This notebook is for downloading and clipping raw data to nc
For downloading raw data, it queries APIs.  
For preprocessing, it currently clips to an NC mask (data/processed/nc_boundary.gpkg)


In [ ]:
import geopandas as gpd
import earthaccess
from collections import defaultdict
import geopandas as gpd
import rioxarray
from rioxarray.merge import merge_arrays
from peatfire import data_path
from peatfire.data_loading import clip_vector_to_mask, clip_raster_to_mask, get_key
from pathlib import Path
import xarray as xr
import pandas as pd, requests, time
from datetime import date, timedelta
import rasterio
import io
import ee
import geemap

### Raw data downloads

nc boundary download

In [15]:
url = 'https://www2.census.gov/geo/tiger/GENZ2018/shp/cb_2018_us_state_500k.zip'
states = gpd.read_file(url)            # geopandas reads the zip directly
nc = states[states['NAME'] == 'North Carolina']
nc.to_file('../data/processed/boundaries/nc_boundary.gpkg', driver='GPKG')   # GeoPackage > shapefile

earthaccess bulk download of MCD64A1

In [2]:
earthaccess.login()  # uses Earthdata credentials / .netrc

results = earthaccess.search_data(
    short_name='MCD64A1',
    version='061',
    temporal=('2000-11-01', '2026-06-03'), # set whatever dates you want - just do about a year for now for testing
    bounding_box=(-84.322, 33.842, -75.461, 36.588),  # NC bbox; returns h11v05 + h12v05
)
earthaccess.download(results, '../data/raw/fire/MCD64A1_061/')

/Users/jinjiang-macair/anaconda3/envs/peat_fire_stanback/lib/python3.11/site-packages/earthaccess/results.py:343: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  self["size"] = self.size()
/Users/jinjiang-macair/anaconda3/envs/peat_fire_stanback/lib/python3.11/site-packages/earthaccess/store.py:832: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  total_size = round(sum(granule.size() for granule in granules) / 1024, 2)
QUEUEING TASKS | : 100%|██████████| 610/610 [00:00<00:00, 77745.46it/s]
PROCESSING TASKS | : 100%|██████████| 610/610 [01:51<00:00,  5.49it/s]
COLLECTING RESULTS | : 100%|██████████| 610/610 [00:00<00:00, 1685458.13it/s]


[PosixPath('../data/raw/fire/MCD64A1_061/MCD64A1.A2000306.h11v05.061.2021307220204.hdf'),
 PosixPath('../data/raw/fire/MCD64A1_061/MCD64A1.A2000306.h10v05.061.2021307220206.hdf'),
 PosixPath('../data/raw/fire/MCD64A1_061/MCD64A1.A2000336.h11v05.061.2021307220306.hdf'),
 PosixPath('../data/raw/fire/MCD64A1_061/MCD64A1.A2000336.h10v05.061.2021307220309.hdf'),
 PosixPath('../data/raw/fire/MCD64A1_061/MCD64A1.A2001001.h10v05.061.2021307220407.hdf'),
 PosixPath('../data/raw/fire/MCD64A1_061/MCD64A1.A2001001.h11v05.061.2021307220405.hdf'),
 PosixPath('../data/raw/fire/MCD64A1_061/MCD64A1.A2001032.h10v05.061.2021307220531.hdf'),
 PosixPath('../data/raw/fire/MCD64A1_061/MCD64A1.A2001032.h11v05.061.2021307220534.hdf'),
 PosixPath('../data/raw/fire/MCD64A1_061/MCD64A1.A2001060.h11v05.061.2021307220624.hdf'),
 PosixPath('../data/raw/fire/MCD64A1_061/MCD64A1.A2001060.h10v05.061.2021307220624.hdf'),
 PosixPath('../data/raw/fire/MCD64A1_061/MCD64A1.A2001091.h10v05.061.2021307220732.hdf'),
 PosixPath

ee download of GABAM GeoTIFF images clipped to NC

In [ ]:
ee_id = str(get_key(ee_project_id))
ee_id

In [ ]:

ee.Authenticate()
ee.Initialize(project=str(get_key(ee_id)))

### Clipping to NC

first grab the NC bounds shapefile

In [3]:
nc = gpd.read_file(data_path('processed', 'boundaries', 'nc_boundary.gpkg'))

clip MCD64A1 to NC

In [8]:
def sds(hdf):
    """Return the GDAL subdataset string for the Burn Date layer of an MCD64A1 file."""
    with rasterio.open(str(hdf)) as src:
        for name in src.subdatasets:
            if name.rstrip().endswith("Burn Date"):
                return name
    raise ValueError(f"No 'Burn Date' subdataset found in {hdf.name}")

# 2. group the two tiles by acquisition date (filename token A2017001, A2017032, ...)
raw = data_path('raw', 'fire', 'MCD64A1_061')
by_date = defaultdict(list)
for f in sorted(raw.glob("MCD64A1.*.hdf")):
    by_date[f.name.split(".")[1]].append(f)

# 3. mosaic -> clip -> save, per date
out = data_path('processed', 'fire', 'MCD64A1_061')
out.mkdir(parents=True, exist_ok=True)

for date, files in by_date.items():
    tiles = [rioxarray.open_rasterio(sds(f), masked=True) for f in files]
    mosaic = merge_arrays(tiles) # stitch h10v05 + h11v05
    nc_sin = nc.to_crs(mosaic.rio.crs) # reproject NC -> sinusoidal
    clip = mosaic.rio.clip(nc_sin.geometry, nc_sin.crs, drop=True)
    clip.rio.to_raster(out / f"MCD64A1_{date}_nc.tif")
    

/Users/jinjiang-macair/anaconda3/envs/peat_fire_stanback/lib/python3.11/site-packages/rasterio/__init__.py:356: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, **kwargs)
/Users/jinjiang-macair/anaconda3/envs/peat_fire_stanback/lib/python3.11/site-packages/rasterio/__init__.py:356: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, **kwargs)
/Users/jinjiang-macair/anaconda3/envs/peat_fire_stanback/lib/python3.11/site-packages/rasterio/__init__.py:356: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, **kwargs)
/Users/jinjiang-macair/anaconda3/envs/peat_fire_stanback/lib/python3.11/site-packages/rasterio/__init__.py:356: NotGeoref

clip VIIRS to NC

In [ ]:
viirs_snpp_archive_nc = clip_vector_to_mask(data_path('raw', 'fire', 'DL_FIRE_SV-C2_758389', 'fire_archive_SV-C2_758389.shp'), nc, data_path('processed', 'fire', 'viirs', f'viirs_snpp_archive_nc.gpkg'))
viirs_noaa20_nrt_nc = clip_vector_to_mask(data_path('raw', 'fire', 'DL_FIRE_J1V-C2_758431', 'fire_nrt_J1V-C2_758431.shp'), nc, data_path('processed', 'fire', 'viirs', f'viirs_noaa20_nrt_nc.gpkg'))
viirs_noaa21_nrt_nc = clip_vector_to_mask(data_path('raw', 'fire', 'DL_FIRE_J2V-C2_758399', 'fire_nrt_J2V-C2_758399.shp'), nc, data_path('processed', 'fire', 'viirs', f'viirs_noaa21_nrt_nc.gpkg'))

clip SE firemap to NC

In [4]:
for year in range(2000, 2023):
    _ = clip_raster_to_mask(data_path('raw', 'fire', 'se_firemap', f'cbi_mosaic_{str(year)}', f'cbi_mosaic_{str(year)}.tif'), nc, data_path('processed', 'fire', 'se_firemap', f'cbi_mosaic_{str(year)}_nc', f'cbi_mosaic_{str(year)}_nc.tif'))